In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from cvxopt import matrix, solvers

In [2]:
# Silence the internal CVXOPT solver progress outputs
solvers.options['show_progress'] = False

class LinearSVM:
    def __init__(self, C=1.0):
        self.C = C
        self.w = None
        self.b = None

    def fit(self, X, y):
        """Train the SVM using Quadratic Programming (QP)."""
        n_samples, n_features = X.shape

        # Compute the kernel matrix
        K = np.dot(X, X.T)

        # Set up QP parameters
        P = matrix(np.outer(y, y) * K, tc='d')
        q = matrix(-np.ones(n_samples), tc='d')
        G = matrix(np.vstack((-np.eye(n_samples), np.eye(n_samples))), tc='d')
        h = matrix(np.hstack((np.zeros(n_samples), np.ones(n_samples) * self.C)), tc='d')
        A = matrix(y.reshape(1, -1), tc='d')
        b = matrix(0.0, tc='d')

        # Solve the QP problem
        sol = solvers.qp(P, q, G, h, A, b)
        alpha = np.ravel(sol['x'])

        # Identify support vectors (alpha > 1e-5)
        sv_mask = alpha > 1e-5
        self.alpha = alpha[sv_mask]
        self.support_vectors = X[sv_mask]
        self.support_labels = y[sv_mask]

        # Calculate weights
        self.w = np.sum(self.alpha[:, None] * self.support_labels[:, None] * self.support_vectors, axis=0)

        # Calculate bias strictly using support vectors that lie directly on the margin (0 < alpha < C)
        margin_sv = (alpha > 1e-5) & (alpha < self.C - 1e-5)
        if np.any(margin_sv):
            self.b = np.mean(y[margin_sv] - np.dot(X[margin_sv], self.w))
        else:
            self.b = np.mean(self.support_labels - np.dot(self.support_vectors, self.w))

    def predict(self, X):
        """Predict labels (-1 or 1) for the input data."""
        return np.sign(np.dot(X, self.w) + self.b)

In [3]:
def load_data(file_path, num_samples=5000):
    """Load dataset, limit samples, and binarize labels to (-1, 1)."""
    data = pd.read_csv(file_path)
    X = data.iloc[:num_samples, :-1].values
    y = np.where(data.iloc[:num_samples, -1] == 0, -1, 1)
    return X, y

In [4]:
def plot_decision_boundary(X, y, model):
    """Plot the decision boundary for 2D classification data."""
    plt.figure(figsize=(8, 6))
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', alpha=0.5, edgecolors='k')

    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contour(xx, yy, Z, levels=[0], colors='black', linewidths=2)

    plt.title("SVM Decision Boundary (2D PCA)")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.show()


In [ ]:
if __name__ == '__main__':
    # Load and split dataset
    X, y = load_data('mnist_784.csv')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Standardize features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Reduce dimensions to 2D for plotting visualization
    pca = PCA(n_components=2)
    X_train_2d = pca.fit_transform(X_train)
    X_test_2d = pca.transform(X_test)

    # Train Custom SVM
    svm = LinearSVM(C=1.0)
    svm.fit(X_train_2d, y_train)

    # Evaluate Performance
    y_pred = svm.predict(X_test_2d)

    print("=" * 25)
    print("      TEST METRICS       ")
    print("=" * 25)
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.2%}")
    print(f"Precision: {precision_score(y_test, y_pred):.2%}")
    print(f"Recall:    {recall_score(y_test, y_pred):.2%}")
    print(f"F1 Score:  {f1_score(y_test, y_pred):.2%}")
    print("=" * 25)

    # Render Visual Plot
    plot_decision_boundary(X_train_2d, y_train, svm)